# 31 – LODES Jobs Aggregation (2022)

This notebook processes **LODES workplace area characteristics (WAC)** data to compute total jobs (and optionally by sector) for each spatial unit (block or block group).

**Goals:**
- Load 2022 LODES WAC file(s) for Arizona.
- Filter to the relevant county / region.
- Aggregate job counts by GEOID.
- Save a table of jobs per GEOID for spatial joining.


In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
JOBS_DIR = DATA_DIR / "jobs"

JOBS_DIR

WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/jobs')

In [ ]:
wac_s000_path = JOBS_DIR / "lodes_raw" / "az_wac_S000_JT00_2022.csv"

wac_s000_path

WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/jobs/lodes_raw/az_wac_S000_JT00_2022.csv')

In [ ]:
wac_s000 = pd.read_csv(wac_s000_path, dtype={"w_geocode": "string"})

print("Rows:", len(wac_s000))
wac_s000.head()

Rows: 36998


,w_geocode,C000,CA01,CA02,CA03,CE01,CE02,CE03,CNS01,CNS02,...,CFA02,CFA03,CFA04,CFA05,CFS01,CFS02,CFS03,CFS04,CFS05,createdate
0,040019426001161,13,4,8,1,4,6,3,0,0,...,0,0,0,0,0,0,0,0,0,20240920
1,040019427001045,9,4,2,3,0,8,1,0,0,...,0,0,0,0,0,0,0,0,0,20240920
2,040019427001046,134,13,68,53,4,25,105,0,0,...,0,0,0,0,0,0,0,0,0,20240920
3,040019427002097,7,0,3,4,0,1,6,0,0,...,0,0,0,0,0,0,0,0,0,20240920
4,040019427002134,88,12,46,30,20,62,6,0,0,...,0,0,0,0,0,0,0,0,0,20240920


In [8]:
wac_s000.columns

Index(['w_geocode', 'C000', 'CA01', 'CA02', 'CA03', 'CE01', 'CE02', 'CE03',
       'CNS01', 'CNS02', 'CNS03', 'CNS04', 'CNS05', 'CNS06', 'CNS07', 'CNS08',
       'CNS09', 'CNS10', 'CNS11', 'CNS12', 'CNS13', 'CNS14', 'CNS15', 'CNS16',
       'CNS17', 'CNS18', 'CNS19', 'CNS20', 'CR01', 'CR02', 'CR03', 'CR04',
       'CR05', 'CR07', 'CT01', 'CT02', 'CD01', 'CD02', 'CD03', 'CD04', 'CS01',
       'CS02', 'CFA01', 'CFA02', 'CFA03', 'CFA04', 'CFA05', 'CFS01', 'CFS02',
       'CFS03', 'CFS04', 'CFS05', 'createdate'],
      dtype='object')

In [9]:
print(wac_s000["w_geocode"].str.len().value_counts())

w_geocode
15    36998
Name: count, dtype: Int64


In [13]:
# since GEOIDs will start with state and county FIPS, we can filter using startswith to get Maricopa County (state code 04 + county code 013)

wac_maricopa = wac_s000[wac_s000["w_geocode"].str.startswith("04013")].copy()
print(len(wac_s000), "rows total,", len(wac_maricopa), "rows in Maricopa")

36998 rows total, 22090 rows in Maricopa


In [14]:
job_cols = ["C000", "CE01", "CE02", "CE03"]  # total + earnings tiers

# Group by block group and sum jobs
wac_jobs_maricopa = (
    wac_maricopa
    .groupby("w_geocode", as_index=False)[job_cols]
    .sum()
)

print(wac_jobs_maricopa.head())

         w_geocode  C000  CE01  CE02  CE03
0  040130101021045    11     3     6     2
1  040130101021049     1     1     0     0
2  040130101021098     9     2     3     4
3  040130101021114     4     0     2     2
4  040130101021116     1     0     1     0


In [ ]:
wac_jobs_maricopa.to_csv(JOBS_DIR / "lodes_processed" / "lodes_jobs_maricopa_2022.csv", index=False)
print("Saved cleaned LODES spatial units to:", JOBS_DIR / "lodes_processed" / "lodes_jobs_maricopa_2022.csv")